# Quickstart: iterative DPO on Qwen3-235B-A22B-Instruct-2507

Required API keys (in `.env`):

- `TINKER_API_KEY`
- `OPENAI_API_KEY`, `OPENROUTER_API_KEY` (nl_gameable graders)
- `ANTHROPIC_API_KEY` (monitor-disruption judge)
- `WANDB_API_KEY` (optional; set `WANDB_PROJECT = None` to skip)

In [ ]:
import json
import os
import sys
from dataclasses import asdict, replace
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()  # run_select / the tinker DPO trainer call asyncio.run()

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "rewardhacking_training").exists():
    assert REPO_ROOT != REPO_ROOT.parent, "run from inside the repo"
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
os.environ["PATH"] = f"{REPO_ROOT / '.venv' / 'bin'}:{os.environ['PATH']}"  # impossible_mbpp shells out to `python`
os.environ.setdefault("INSPECT_DISPLAY", "plain")
load_dotenv(REPO_ROOT / ".env")
for key in ("TINKER_API_KEY", "OPENAI_API_KEY", "OPENROUTER_API_KEY", "ANTHROPIC_API_KEY"):
    assert os.environ.get(key), f"{key} not set"

from rewardhacking_training.data_sampling import dataset_prompt_ids, round_prompt_ids
from rewardhacking_training.envs.nl_gameable.nl_gameable_env import DEFAULT_STANDARDIZE_STATS_PATH
from rewardhacking_training.generate.generate import GenerateConfig, ModelConfig
from rewardhacking_training.generate.inference_client import InferenceClient, InferenceClientConfig
from rewardhacking_training.select.select import SelectConfig
from rewardhacking_training.train.train import TrainConfig
from rewardhacking_training.training_iteration import (
    Iteration, combine_stage, generate_stage, run_iteration, select_stage, stage_result, train_stage,
)

# ---- hyperparameters (2026-09-02_qwen235b_dpo_8k, arm lr2e4-bs8-8k) ----------
MODEL = "Qwen/Qwen3-235B-A22B-Instruct-2507"
RENDERER = "qwen3_instruct_thinking"  # qwen3_instruct with trained reasoning spelled <thinking>...</thinking> (rewardhacking_training/train/train_providers/tinker/renderers.py)
N_ITERATIONS = 3
SEED = 42

N_SAMPLES = 50            # completions per prompt
MAX_TOKENS = 8192         # generation completion cap
MAX_TOTAL_TOKENS = 12288  # select: drop rendered prompt+completion longer than this
MAX_CONNECTIONS = 500
ATTEMPT_TIMEOUT = 1800    # s per sample; 8k completions at 500-way concurrency need it

# Instruct-2507 is not a native reasoner: reasoning is prompted with <thinking>
# tags (a literal <think> token in the system prompt is OOD for it) and parsed
# by the envs' extract_thinking solver.
SYSTEM_PROMPTS_PATH = "rewardhacking_training/prompts/system_prompts/thinking_variants_no_think.json"
ENV_TASKS = {
    "nl_gameable": "rewardhacking_training.envs.nl_gameable.nl_gameable_env:nl_gameable",
    "impossible_mbpp": "rewardhacking_training.envs.impossible_mbpp.impossible_mbpp_env:impossible_mbpp",
}
ENV_TASK_ARGS = {
    "nl_gameable": {
        "max_tokens": MAX_TOKENS,
        "scorer_mode": "programmatic",
        "standardize_stats_path": DEFAULT_STANDARDIZE_STATS_PATH,  # per-prompt z vs gpt-4.1-mini k=16 teacher
    },
    "impossible_mbpp": {"max_tokens": MAX_TOKENS},
}
EPOCH_FRACTION = {"nl_gameable": 0.5, "impossible_mbpp": 1.0}  # of each env's prompt epoch per iteration
ENV_SELECT_ARGS = {
    "nl_gameable": {"n": 2},
    "impossible_mbpp": {"n": 6, "dpo_pairing": "score_diversity", "n_per_score_pair": 2, "score_threshold": 1.0},
}
ENVS = list(ENV_TASKS)

LEARNING_RATE = 2e-4
BATCH_SIZE = 8
BETA = 0.1
LORA_RANK = 32
N_EPOCHS = 1
LR_SCHEDULE = "cosine"
WARMUP_STEPS = 10
WANDB_PROJECT = "qwen235b_itdpo_quickstart"  # None disables W&B

RUN_NAME = "qwen235b_itdpo_8k"
RUN_DIR = REPO_ROOT / "output" / "notebooks" / "qwen235b_iterative_dpo_quickstart" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"run dir: {RUN_DIR}")

## Per-iteration configs

In [ ]:
TINKER = InferenceClientConfig(provider="tinker", base_model=MODEL, tinker_renderer_name=RENDERER)


def iter_dir(i: int) -> Path:
    return RUN_DIR / f"iter_{i:02d}"


def prompt_ids(env: str, i: int) -> list[str]:
    return round_prompt_ids(
        dataset_prompt_ids(ENV_TASKS[env], ENV_TASK_ARGS[env]), i,
        seed=SEED, name=env, epoch_fraction=EPOCH_FRACTION[env],
    )


def generate_cfg(env: str, i: int) -> GenerateConfig:
    return GenerateConfig(
        task=ENV_TASKS[env],
        model_config=ModelConfig(inference_client=TINKER),  # include_reasoning=False: <thinking> stays inline
        n_samples=N_SAMPLES,
        system_prompts_path=SYSTEM_PROMPTS_PATH,
        task_args={**ENV_TASK_ARGS[env], "prompt_ids": prompt_ids(env, i)},
        max_connections=MAX_CONNECTIONS,
        attempt_timeout=ATTEMPT_TIMEOUT,
        retry_on_failure=False,  # a partial tinker generation can't be resumed sample-by-sample
    )


def select_cfg(env: str) -> SelectConfig:
    return SelectConfig(
        seed=SEED, max_total_tokens=MAX_TOTAL_TOKENS, token_count_model=MODEL, **ENV_SELECT_ARGS[env],
    )


def train_cfg() -> TrainConfig:
    return TrainConfig(
        provider="tinker",
        base_model=MODEL,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        beta=BETA,
        lora_rank=LORA_RANK,
        n_epochs=N_EPOCHS,
        tinker_renderer_name=RENDERER,
        tinker_lr_schedule=LR_SCHEDULE,
        tinker_warmup_steps=WARMUP_STEPS,
        wandb_project=WANDB_PROJECT,
    )


def iteration(i: int) -> Iteration:
    prev = None if i == 0 else stage_result(iter_dir(i - 1))
    assert i == 0 or prev is not None, f"iteration {i - 1} has not finished training"
    return Iteration(
        stage_dir=iter_dir(i),
        method="dpo",
        model=MODEL if i == 0 else prev["model"],
        generate={env: generate_cfg(env, i) for env in ENVS},
        select={env: select_cfg(env) for env in ENVS},
        train=train_cfg(),
        suffix=f"{RUN_NAME}-it{i:02d}",
        resume_handle=None if i == 0 else prev["resume_handle"],
        seed=SEED,
    )


def show_config(cfg, title: str | None = None) -> None:
    if title:
        print(f"=== {title}")
    for k, v in asdict(cfg).items():
        if k.startswith(("openai_", "together_", "modal_")):  # other providers' knobs
            continue
        if k == "task_args" and "prompt_ids" in v:
            v = {**v, "prompt_ids": f"[{len(v['prompt_ids'])} ids]"}
        print(f"  {k} = {v}")


def n_lines(p: Path) -> int:
    return sum(1 for line in p.open() if line.strip())

## Iteration 0, stage by stage
### Generate + score

In [ ]:
it0 = iteration(0)
for env in ENVS:
    show_config(it0.generate[env], f"generate / {env}")

In [ ]:
for env in ENVS:
    generate_stage(it0.stage_dir, env, replace(it0.generate[env], model=it0.model))  # force=True to regenerate

### Select + combine

In [ ]:
for env in ENVS:
    show_config(it0.select[env], f"select / {env}")

In [ ]:
for env in ENVS:
    path = select_stage(it0.stage_dir, env, "dpo", it0.select[env])  # force=True to reselect
    print(f"{env}: {n_lines(path)} pairs")
dpo_path = combine_stage(it0.stage_dir, "dpo", ENVS, seed=SEED)
print(f"combined: {n_lines(dpo_path)} pairs -> {dpo_path}")

### Preview the pairs

In [ ]:
from utils.dpo_widget import dpo_pairs_widget

dpo_pairs_widget(dpo_path)

### Train

In [ ]:
train_config = replace(it0.train, method="dpo", model=it0.model, resume_handle=None, suffix=it0.suffix, wandb_name=it0.suffix)
show_config(train_config, "train")
print(f"\n{n_lines(dpo_path)} pairs -> ~{-(-n_lines(dpo_path) // BATCH_SIZE) * N_EPOCHS} steps")

In [ ]:
result = train_stage(it0.stage_dir, train_config)  # force=True to retrain
print(result["model"])

## Iterations 1–2

In [ ]:
for i in range(1, N_ITERATIONS):
    print(f"\n========== iteration {i} ==========")
    result = run_iteration(iteration(i))
    print(f"[iter {i}] -> {result['model']}")

final = stage_result(iter_dir(N_ITERATIONS - 1))
(RUN_DIR / "final_model.txt").write_text(final["model"] + "\n")
print(f"\nfinal model: {final['model']}")

### Training-slice generation metrics

In [ ]:
from experiment_utils.training_curves import gen_metrics
from rewardhacking_training.envs.nl_gameable.nl_gameable_env import load_standardize_stats
from rewardhacking_training.training_iteration import gen_dir

nlg_stats = load_standardize_stats(DEFAULT_STANDARDIZE_STATS_PATH)
print(f"{'iter':>4}  {'sampled from':<8} {'mbpp pass-all':>14} {'mbpp mean':>10} {'nlg mean z':>11} {'nlg raw':>8}")
for i in range(N_ITERATIONS):
    mbpp = gen_metrics(gen_dir(iter_dir(i), "impossible_mbpp"), "coding")
    nlg = gen_metrics(gen_dir(iter_dir(i), "nl_gameable"), "nlg", nlg_stats)
    src = "base" if i == 0 else f"it{i - 1:02d}"
    print(f"{i:>4}  {src:<8} {mbpp['passall_rate']:>14.3f} {mbpp['mean_score']:>10.3f} {nlg['mean_z']:>11.2f} {nlg['mean_raw']:>8.1f}")

## Evaluation

In [ ]:
import inspect_ai

from inspect_ai.log import read_eval_log
from experiment_utils.metrics import binom_se, hack_rate, header_metrics, latest_eval, mis_rate
from experiment_utils.plotting import PALETTE, checkpoint_ticks, use_style
import matplotlib.pyplot as plt

EVAL_LOGS = RUN_DIR.parent / "eval_logs" / RUN_NAME
EVAL_MAX_TOKENS = 8192
EVAL_TEMPERATURE = 1.0
EVAL_MAX_CONNECTIONS = 200
COT = dict(use_cot=True, is_native_reasoning_model=False)

CHECKPOINTS = [("base", MODEL)] + [
    (f"it{i:02d}", stage_result(iter_dir(i))["model"])
    for i in range(N_ITERATIONS) if stage_result(iter_dir(i)) is not None
]
LABELS = [label for label, _ in CHECKPOINTS]
for label, model in CHECKPOINTS:
    print(f"{label:>5} = {model}")


def eval_done(cell_dir: Path) -> bool:
    # a failed / interrupted eval also leaves a .eval behind; only a successful one counts
    log = latest_eval(cell_dir)
    return log is not None and read_eval_log(str(log), header_only=True).status == "success"


def run_eval(cell: str, make_task, **eval_kwargs) -> None:
    '''`make_task()` builds the inspect task; runs it on every checkpoint whose cell is not done.'''
    for label, model in CHECKPOINTS:
        cell_dir = EVAL_LOGS / label / cell
        if eval_done(cell_dir):
            print(f"--- {label}/{cell}: done, skipping")
            continue
        print(f"--- {label}/{cell}: running on {model}")
        client = InferenceClient(TINKER)
        inspect_model, model_args = client.start(model)
        inspect_model.config.max_tokens = EVAL_MAX_TOKENS  # these tasks take no max_tokens param
        inspect_model.config.max_connections = EVAL_MAX_CONNECTIONS
        try:
            inspect_ai.eval(
                make_task(), model=inspect_model, model_args=model_args or {}, log_dir=str(cell_dir),
                max_connections=EVAL_MAX_CONNECTIONS, retry_on_error=3, fail_on_error=0.1, **eval_kwargs,
            )
        finally:
            client.end()


def ladder_plot(values, ylabel: str, title: str, ylim=(0, 100)) -> None:
    '''One errorbar curve over the checkpoint ladder; `values[label]` = (rate, se) in [0, 1] or None.'''
    use_style()
    pts = [(x, v) for x, lbl in enumerate(LABELS) if (v := values.get(lbl)) is not None]
    fig, ax = plt.subplots(figsize=(1.6 + 0.7 * len(LABELS), 3.2))
    ax.errorbar([x for x, _ in pts], [100 * v for _, (v, _) in pts], yerr=[100 * se for _, (_, se) in pts],
                color=PALETTE[1], marker="o", linewidth=2, capsize=2.5)
    checkpoint_ticks(ax, LABELS)
    ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.tight_layout()
    plt.show()
    for x, (v, se) in pts:
        print(f"  {LABELS[x]:>5}: {100 * v:5.1f} ± {100 * se:.1f}")

### Toy reward

In [ ]:
from rewardhacking_evals.toy_reward import toy_reward_eval

run_eval("toy_reward_noinstr_exp", lambda: toy_reward_eval(
    instruct_no_hack=False, grader_line="explicit_odd", n_samples_per_combination=10, temperature=EVAL_TEMPERATURE, **COT,
))

In [ ]:
ladder_plot({lbl: hack_rate(EVAL_LOGS / lbl / "toy_reward_noinstr_exp") for lbl in LABELS},
            ylabel="gaming rate (%)", title="toy reward (no instruction, explicit-odd grader)")

### Monitor disruption

In [ ]:
from misalignment_evals.monitor_disruption import monitor_disruption_eval

run_eval("mis_monitor_disruption", lambda: monitor_disruption_eval())

In [ ]:
ladder_plot({lbl: mis_rate(EVAL_LOGS / lbl / "mis_monitor_disruption") for lbl in LABELS},
            ylabel="misaligned rate (%)", title="monitor disruption")

### IFEval

In [ ]:
from capabilities_evals.ifeval import ifeval_eval

run_eval("cap_ifeval", lambda: ifeval_eval(temperature=EVAL_TEMPERATURE, max_tokens=EVAL_MAX_TOKENS, **COT))

In [ ]:
def ifeval_acc(label: str):
    m = header_metrics(EVAL_LOGS / label / "cap_ifeval")
    return None if m is None else (m["prompt_strict_acc"], binom_se(m["prompt_strict_acc"], m["_n"]))


ladder_plot({lbl: ifeval_acc(lbl) for lbl in LABELS}, ylabel="prompt-level strict accuracy (%)",
            title="IFEval", ylim=(50, 100))